# touster

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Patan-Sameer66/touster/blob/main/touster.ipynb)

Fine-tune a small model for one task using a large teacher model.

Run the cells in order, or use **Run All**. Each cell saves its results to the run folder and skips work that is
already done, so after a Colab disconnect you can run everything again and it picks up where it stopped.
To redo one cell, set `FORCE = True` in that cell.

Cells 1-3 work without a GPU when the teacher is a cloud API or a local Ollama server.
Cells 4-7 need an NVIDIA GPU (Colab: Runtime > Change runtime type > T4 GPU).

## 1. Config and setup

Edit `CONFIG` below; it is the only place you need to change to switch teacher, student or task.
Optionally point `CONFIG_YAML` at a YAML file that overrides any field.

- **Reads:** `CONFIG`, optional YAML file, API keys from environment variables or Colab Secrets.
- **Writes:** `runs/<run_id>/config.json`, `runs/<run_id>/state.json`.

Re-running this cell resumes the newest run for the same `task_name`. Set `NEW_RUN = True` to start over.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Patan-Sameer66/touster.git"
if "google.colab" in sys.modules:
    if not os.path.isfile("requirements.txt"):
        if not os.path.isdir("touster-repo"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "touster-repo"], check=True)
        os.chdir("touster-repo")
    from touster import install_requirements
    install_requirements()  # first run only; it asks you to restart the runtime if needed

CONFIG = {
    "task_name": "sentiment",
    "task_description": "Classify the sentiment of a short product review. Answer with exactly one word: positive, negative or neutral.",
    "task_type": "classification",
    "verifiable": True,
    # Any LiteLLM model id. Local: "ollama_chat/qwen2.5:3b". Cloud examples: "gpt-4o-mini", "gemini/gemini-2.5-flash".
    "teacher_model": "ollama_chat/qwen2.5:3b",
    "student_model": "unsloth/Qwen2.5-1.5B-Instruct",
    "golden_path": "examples/golden_sample.jsonl",
    "budget_mode": "money",
    "max_usd": 5.0,
    "run_dir": "runs",  # on Colab use a mounted Drive path, e.g. "/content/drive/MyDrive/touster_runs", to survive disconnects
    "smoke_test": False,
}
CONFIG_YAML = None
NEW_RUN = False

from touster import setup_run

run = setup_run(CONFIG, CONFIG_YAML, new_run=NEW_RUN)

## 2. Golden dataset

Loads your hand-written examples, checks them, and splits them into a seed set (used to guide generation and
training) and an eval set (used only for scoring, never for training).

- **Reads:** `golden_path`.
- **Writes:** `data/golden_seed.jsonl`, `data/golden_eval.jsonl`, eval ids in `state.json`.

In [ ]:
from touster.data.golden import run_golden_cell

FORCE = False
run_golden_cell(run.cfg, run.paths, force=FORCE)

## 3. Synthetic data and calibration

Runs a small calibration first and shows the estimated cost and time for the full run, then asks you to confirm.
The teacher writes new prompts from the seeds, evolves them, and answers them. Filters then remove schema errors,
near-duplicates, low-quality items and anything overlapping the eval set.

- **Reads:** `data/golden_seed.jsonl`, `data/golden_eval.jsonl` (decontamination only).
- **Writes:** `data/synthetic_raw.jsonl`, `data/synthetic_clean.jsonl`, `costs.jsonl`.

In [ ]:
from touster.data.generate import run_synthetic_cell

FORCE = False
run_synthetic_cell(run.cfg, run.paths, run.client, run.budget, force=FORCE)

## 4. Supervised fine-tuning (QLoRA)

Loads the student in 4-bit and trains LoRA adapters on the clean synthetic data plus the seed set.
Checkpoints are saved every 200 steps; re-running continues from the last one. Needs a GPU.

- **Reads:** `data/synthetic_clean.jsonl`, `data/golden_seed.jsonl`.
- **Writes:** `checkpoints/sft/`.

In [ ]:
from touster.train.sft import run_sft_cell

FORCE = False
run_sft_cell(run.cfg, run.paths, force=FORCE)

## 5. Evaluation

Scores the base model, then the fine-tuned model, on the eval set: task metric, pairwise judge against the base
(each pair judged twice with the order swapped), and a small general-knowledge regression check. Needs a GPU.

- **Reads:** `data/golden_eval.jsonl`, `checkpoints/sft/final`.
- **Writes:** `eval/base.json`, `eval/sft.json`.

In [ ]:
from touster.eval.metrics import run_eval_cell

FORCE = False
run_eval_cell(run.cfg, run.paths, run.client, run.budget, force=FORCE)

## 6. Self-improvement loop (optional)

Each round samples several answers per training prompt, has the judge score them, and trains with DPO on
best-vs-worst pairs (or GRPO with a programmatic reward when `loop_method="grpo"`). Stops when gains stall,
`max_rounds` is reached or the budget runs out. The best checkpoint is kept, never a worse later one.
Skipped when `self_improve` is false. Needs a GPU.

- **Reads:** training data, the current best adapter.
- **Writes:** `data/dpo_round_<n>.jsonl`, `checkpoints/dpo_round_<n>/`, `eval/dpo_round_<n>.json`.

In [ ]:
from touster.train.dpo import run_loop_cell

FORCE = False
run_loop_cell(run.cfg, run.paths, run.client, run.budget, force=FORCE)

## 7. Export and deploy

Merges the best adapter into the base weights, exports GGUF, writes an Ollama `Modelfile` and a model card,
optionally pushes to the Hugging Face Hub, and checks that the GGUF loads and answers a prompt.
Set `LAUNCH_DEMO = True` for a Gradio page comparing base and fine-tuned answers. Needs a GPU.

- **Reads:** the best adapter, `eval/*.json`, `costs.jsonl`.
- **Writes:** `export/merged/`, `export/gguf/`, `export/Modelfile`, `export/MODEL_CARD.md`.

In [ ]:
from touster.export import launch_demo, run_export_cell

FORCE = False
LAUNCH_DEMO = False
run_export_cell(run.cfg, run.paths, force=FORCE)
if LAUNCH_DEMO:
    launch_demo(run.cfg, run.paths)

## 8. Report

Builds a self-contained HTML report (charts work offline) and shows it below.

- **Reads:** everything in the run folder.
- **Writes:** `report.html`.

In [ ]:
from IPython.display import HTML, display
from touster.report import report_html, run_report_cell

FORCE = False
run_report_cell(run.cfg, run.paths, force=FORCE)
display(HTML(report_html(run.paths)))